In [3]:
# create a df to demo some scenarios
person_list = [(100, "Prashant", 30),
             (101, "David", None),
             (102, "Sushant",None),
             (103, "Abdul", 45),
             (104, "Shruti", 28)]

person_df = spark.createDataFrame(person_list).toDF("id", "name", "age")
person_df.show()

+---+--------+----+
| id|    name| age|
+---+--------+----+
|100|Prashant|  30|
|101|   David|NULL|
|102| Sushant|NULL|
|103|   Abdul|  45|
|104|  Shruti|  28|
+---+--------+----+



In [20]:
"""
Find all records where age is 28.
Find all records where age is not given or unknown.
"""
from pyspark.sql.functions import col

# filter
age_filter_df = person_df.filter(col("age") == 28)
age_filter_df.show()

# NULL age values
age_null_df = person_df.filter(col("age").isNull())
age_null_df.show()

# non NULL age values
age_not_null_df = person_df.filter(col("age").isNotNull())
age_not_null_df.show()

+---+------+---+
| id|  name|age|
+---+------+---+
|104|Shruti| 28|
+---+------+---+

+---+-------+----+
| id|   name| age|
+---+-------+----+
|101|  David|NULL|
|102|Sushant|NULL|
+---+-------+----+

+---+--------+---+
| id|    name|age|
+---+--------+---+
|100|Prashant| 30|
|103|   Abdul| 45|
|104|  Shruti| 28|
+---+--------+---+



In [19]:
"""
How operators work on NULL values
"""
from pyspark.sql.functions import expr

person_df.withColumn("age_gt_29", expr("age > 29")).show() # where age is null, the age_gt_29 is also null
# NULL values do not participate in any logical operations

+---+--------+----+---------+
| id|    name| age|age_gt_29|
+---+--------+----+---------+
|100|Prashant|  30|     true|
|101|   David|NULL|     NULL|
|102| Sushant|NULL|     NULL|
|103|   Abdul|  45|     true|
|104|  Shruti|  28|    false|
+---+--------+----+---------+



In [18]:
"""
Find all persons where age is greater than 29
"""

person_df.filter(col("age") > 29).show()
# this will not give persons whose age is NULL

+---+--------+---+
| id|    name|age|
+---+--------+---+
|100|Prashant| 30|
|103|   Abdul| 45|
+---+--------+---+



In [23]:
"""
How mathametical operators work on null.
    Calculate experience for every employee using the following formula.
    experience = age - 23
"""

person_df.withColumn("experience", col("age") - 23).show()
# where age is NULL experience will also be NULL

+---+--------+----+----------+
| id|    name| age|experience|
+---+--------+----+----------+
|100|Prashant|  30|         7|
|101|   David|NULL|      NULL|
|102| Sushant|NULL|      NULL|
|103|   Abdul|  45|        22|
|104|  Shruti|  28|         5|
+---+--------+----+----------+



In [28]:
"""
Calculate the experience knowing that age could be null.
If age is null then assume 23 years for experience calculation.
"""
from pyspark.sql.functions import nvl, lit

# person_df.withColumn("experience", expr("nvl(age, 23) - 23")).show()
person_df.withColumn("experience", nvl(col("age"), lit(23)) - 23).show()

+---+--------+----+----------+
| id|    name| age|experience|
+---+--------+----+----------+
|100|Prashant|  30|         7|
|101|   David|NULL|         0|
|102| Sushant|NULL|         0|
|103|   Abdul|  45|        22|
|104|  Shruti|  28|         5|
+---+--------+----+----------+



In [42]:
"""
What is the average age?
"""
from pyspark.sql.functions import avg

# person_df.selectExpr("avg(age) as average_age").show()
person_df.select(avg(col("age"))).show()
# records with NULL age do not participate

+------------------+
|          avg(age)|
+------------------+
|34.333333333333336|
+------------------+



In [44]:
"""
What if we filter out null values before aggregation
"""

# person_df.filter(col("age").isNotNull()).selectExpr("avg(age) as average_age").show()
person_df.filter(col("age").isNotNull()).select(avg(col("age"))).show()
# it would be the same thing as anyways NULL does not participate in aggregations

+------------------+
|          avg(age)|
+------------------+
|34.333333333333336|
+------------------+

